In [1]:
import cv2
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from skimage import io, color
from skimage.filters import threshold_otsu
from pyspark.sql import SparkSession
import numba

# Initialize Spark Session with GraphFrames package
spark = SparkSession.builder \
    .appName("ImageGraph") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.0-s_2.12") \
    .getOrCreate()


:: loading settings :: url = jar:file:/home/matheus/anaconda3/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/matheus/.ivy2/cache
The jars for the packages stored in: /home/matheus/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8c3e02c3-a99b-49f0-b15d-18f50bffd1ac;1.0
	confs: [default]
	found graphframes#graphframes;0.8.2-spark3.0-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 176ms :: artifacts dl 6ms
	:: modules in use:
	graphframes#graphframes;0.8.2-spark3.0-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-------------------------------

In [ ]:
from pyspark.sql.functions import col, abs as sql_abs
from graphframes import GraphFrame

import cv2
import numpy as np

# Assuming Spark session has been started as shown above
def create_graph_from_image(image):
    rows, cols = image.shape
    vertices = []
    edges = []

    for r in range(rows):
        for c in range(cols):
            vertex_id = r * cols + c
            vertices.append((vertex_id, r, c, int(image[r, c])))

            if r > 0:
                neighbor_id = (r-1) * cols + c
                weight = abs(int(image[r, c]) - int(image[r-1, c]))
                edges.append((vertex_id, neighbor_id, weight))
                
            if c > 0:
                neighbor_id = r * cols + (c-1)
                weight = abs(int(image[r, c]) - int(image[r, c-1]))
                edges.append((vertex_id, neighbor_id, weight))
                
            if r > 0 and c > 0:
                neighbor_id = (r-1) * cols + (c-1)
                weight = abs(int(image[r, c]) - int(image[r-1, c-1]))
                edges.append((vertex_id, neighbor_id, weight))
                
            if r > 0 and c < cols-1:
                neighbor_id = (r-1) * cols + (c+1)
                weight = abs(int(image[r, c]) - int(image[r-1, c+1]))
                edges.append((vertex_id, neighbor_id, weight))

    # Create DataFrames for vertices and edges
    vertices_df = spark.createDataFrame(vertices, ["id", "row", "col", "intensity"])
    edges_df = spark.createDataFrame(edges, ["src", "dst", "weight"])

    # Create GraphFrame
    graph = GraphFrame(vertices_df, edges_df)

    return graph

# Load and preprocess the image
image_path = '/home/matheus/github/natural_artificial_vision/imagens_teste/folhas.png'  # Update with your image path
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image, (100, 100))  # Resize for simplicity

# Create the graph
graph = create_graph_from_image(image)

# Example: Display vertices and edges
print("Vertices:")
graph.vertices.show()

print("Edges:")
graph.edges.show()

# Stop Spark session after processing
spark.stop()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from graphframes import GraphFrame
import cv2
import numpy as np
from itertools import combinations

# Initialize Spark Session with GraphFrames package
spark = SparkSession.builder \
    .appName("ImageGraph") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.0-s_2.12") \
    .getOrCreate()

def create_fully_connected_graph_from_image(image):
    rows, cols = image.shape
    vertices = []
    edges = []

    # Create a vertex for each pixel
    for r in range(rows):
        for c in range(cols):
            vertex_id = r * cols + c
            vertices.append((vertex_id, r, c, int(image[r, c])))

    # Create a fully connected graph by connecting each pair of nodes
    for (r1, c1), (r2, c2) in combinations([(r, c) for r in range(rows) for c in range(cols)], 2):
        vertex_id_1 = r1 * cols + c1
        vertex_id_2 = r2 * cols + c2
        weight = abs(int(image[r1, c1]) - int(image[r2, c2]))
        edges.append((vertex_id_1, vertex_id_2, weight))
        edges.append((vertex_id_2, vertex_id_1, weight))  # Since it's an undirected graph

    # Convert vertices and edges to DataFrames
    vertices_df = spark.createDataFrame(vertices, ["id", "row", "col", "intensity"])
    edges_df = spark.createDataFrame(edges, ["src", "dst", "weight"])

    # Create GraphFrame
    graph = GraphFrame(vertices_df, edges_df)

    return graph

# Load and preprocess the image
image_path = '/home/matheus/github/natural_artificial_vision/imagens_teste/folhas.png'  # Update with your image path
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image, (50, 50))  # Resize to avoid memory issues with full connections

# Create the fully connected graph
graph1 = create_fully_connected_graph_from_image(image)

# Example: Display vertices and edges
print("Vertices:")
graph1.vertices.show()

print("Edges:")
graph1.edges.show()

# Stop Spark session after processing
spark.stop()


In [ ]:
print(graph1)

In [ ]:
def find_borders(G, threshold):
    edges = []
    for (u, v, d) in G.edges(data=True):
        if d['weight'] > threshold:
            edges.append((u, v))
    return edges

In [ ]:
def draw_borders(image, edges):
    for (u, v) in edges:
        image[u] = 255
        image[v] = 255
    return image


In [ ]:
edges = find_borders(graph1.edges,30)

border_folhas = np.zeros_like(image)
border_folhas = draw_borders(border_folhas, edges)